# 09c — Optuna: Hyperparameter Tuning
**Referencias:** Akiba et al. (2019) — Optuna paper · documentación oficial optuna.readthedocs.io

## ¿Por qué Optuna en lugar de GridSearchCV?

| | GridSearchCV | RandomizedSearchCV | **Optuna** |
|---|---|---|---|
| Estrategia | Exhaustiva | Aleatoria | Bayesiana (TPE) |
| Escala | Exponencial en N params | O(n_iter) | Eficiente: se dirige a zonas prometedoras |
| Early stopping | No | No | Sí (pruning) |
| Paralelización | Limitada | Limitada | Nativa (SQLite, Redis, DB) |
| Reproducibilidad | Total | Con seed | Con seed |
| API | sklearn | sklearn | Agnóstica al framework |

## TPE — Tree-structured Parzen Estimator (Bergstra et al. 2011)
Optuna usa TPE por defecto. En lugar de modelar $p(y \mid x)$ (como Gaussian Processes), modela:
$$\text{EI}(x) \propto \frac{l(x)}{g(x)}$$
donde $l(x)$ es la densidad de las configuraciones buenas y $g(x)$ la de las malas. Maximizar $l/g$ dirige la búsqueda hacia zonas prometedoras del espacio de hiperparámetros.

## Pruning — parar trials malos temprano
Optuna puede interrumpir un trial si su desempeño intermedio es peor que los anteriores → ahorra cómputo. Requiere reportar métricas intermedias con `trial.report()` y llamar `trial.should_prune()`.

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_contour,
    plot_slice,
)
import xgboost as xgb
import lightgbm as lgb
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, f1_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(42)

# Dataset: comportamiento de usuarios en plataforma SaaS
n = 3000
sessions       = np.random.randint(1, 30, n)
time_on_site   = np.random.uniform(30, 900, n)
pages          = np.random.uniform(1, 12, n)
days_since_reg = np.random.randint(0, 60, n)
features_used  = np.random.randint(0, 15, n)
errors_seen    = np.random.poisson(1.5, n)
channel        = np.random.choice(['organic', 'paid', 'email', 'direct', 'referral'], n)
device         = np.random.choice(['mobile', 'desktop', 'tablet'], n)
plan           = np.random.choice(['free', 'trial', 'pro', 'enterprise'], n, p=[0.55, 0.25, 0.15, 0.05])
country        = np.random.choice(['US', 'UK', 'DE', 'MX', 'BR', 'other'], n)

logit = (
    -4 + sessions * 0.10 + time_on_site * 0.0018 + pages * 0.15
    - days_since_reg * 0.05 + features_used * 0.22 - errors_seen * 0.3
    + np.where(channel == 'email', 1.0, 0)
    + np.where(device == 'desktop', 0.5, 0)
    + np.where(plan == 'trial', 1.3, np.where(plan == 'pro', 2.1, np.where(plan == 'enterprise', 3.0, 0)))
)
prob = 1 / (1 + np.exp(-logit))
converted = (np.random.uniform(0, 1, n) < prob).astype(int)

num_features = ['sessions', 'time_on_site', 'pages', 'days_since_reg', 'features_used', 'errors_seen']
cat_features = ['channel', 'device', 'plan', 'country']

preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), cat_features),
])

X = pd.DataFrame({
    'sessions': sessions, 'time_on_site': time_on_site.round(0),
    'pages': pages.round(2), 'days_since_reg': days_since_reg,
    'features_used': features_used, 'errors_seen': errors_seen,
    'channel': channel, 'device': device, 'plan': plan, 'country': country,
})[num_features + cat_features]
y = converted

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

print(f'Dataset: {n} filas | Tasa conversión: {y.mean():.2%}')

## 1 — Conceptos clave de Optuna
- **Study**: objeto que representa la optimización completa
- **Trial**: una evaluación con una configuración de hiperparámetros
- **Objective function**: función que recibe un `trial`, define el espacio y retorna la métrica a optimizar
- **Sampler**: algoritmo de búsqueda (default: TPE)
- **Pruner**: decide si parar un trial antes de que termine

## 2 — Ejemplo mínimo: optimizar DecisionTree

In [ ]:
# La objective function define el espacio y retorna la métrica
def objective_dt(trial):
    params = {
        'max_depth':         trial.suggest_int('max_depth', 2, 15),
        'min_samples_leaf':  trial.suggest_int('min_samples_leaf', 1, 50),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 30),
        'criterion':         trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'ccp_alpha':         trial.suggest_float('ccp_alpha', 0.0, 0.05),
    }
    model = DecisionTreeClassifier(**params, random_state=42)
    scores = cross_val_score(model, X_train_enc, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    return scores.mean()

study_dt = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_dt.optimize(objective_dt, n_trials=80, show_progress_bar=False)

print(f'Mejor AUC (CV):    {study_dt.best_value:.4f}')
print('Mejores parámetros:')
for k, v in study_dt.best_params.items():
    print(f'  {k:<25} = {v}')

# Evaluar en test
best_dt = DecisionTreeClassifier(**study_dt.best_params, random_state=42)
best_dt.fit(X_train_enc, y_train)
print(f'\nTest AUC: {roc_auc_score(y_test, best_dt.predict_proba(X_test_enc)[:,1]):.4f}')

## 3 — Tipos de sugerencias en Optuna
Optuna define el espacio de búsqueda con métodos del objeto `trial`:

In [ ]:
# Referencia rápida de tipos de sugerencias
reference = """
trial.suggest_int('param', low, high)            → entero en [low, high]
trial.suggest_int('param', low, high, step=2)    → entero en [low, high] con paso
trial.suggest_int('param', low, high, log=True)  → entero en escala log

trial.suggest_float('param', low, high)          → float en [low, high]
trial.suggest_float('param', 1e-5, 1.0, log=True) → float en escala log (útil para lr)

trial.suggest_categorical('param', ['a','b','c']) → valor de una lista

# Espacios condicionales (params que solo aplican si otro param tiene cierto valor)
model_type = trial.suggest_categorical('model', ['tree', 'boost'])
if model_type == 'boost':
    lr = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
"""
print(reference)

## 4 — Optuna con XGBoost y early stopping

In [ ]:
# Split interno para early stopping dentro de cada trial
X_tr, X_val, y_tr, y_val = train_test_split(X_train_enc, y_train, test_size=0.15, random_state=0)

def objective_xgb(trial):
    params = {
        'n_estimators':      1000,  # alto — el early stopping define el real
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'max_depth':         trial.suggest_int('max_depth', 3, 10),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight':  trial.suggest_int('min_child_weight', 1, 20),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'gamma':             trial.suggest_float('gamma', 0.0, 2.0),
        'early_stopping_rounds': 30,
        'eval_metric': 'auc',
        'random_state': 42,
        'verbosity': 0,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return model.best_score

study_xgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=20)
)
study_xgb.optimize(objective_xgb, n_trials=60, show_progress_bar=False)

print(f'Mejor Val AUC: {study_xgb.best_value:.4f}')
print('Mejores parámetros:')
for k, v in study_xgb.best_params.items():
    print(f'  {k:<25} = {v:.6f}' if isinstance(v, float) else f'  {k:<25} = {v}')

# Reentrenar con todos los datos de train
best_xgb_params = {**study_xgb.best_params, 'n_estimators': 1000,
                   'early_stopping_rounds': 30, 'eval_metric': 'auc',
                   'random_state': 42, 'verbosity': 0}
best_xgb = xgb.XGBClassifier(**best_xgb_params)
best_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
print(f'\nTest AUC: {roc_auc_score(y_test, best_xgb.predict_proba(X_test_enc)[:,1]):.4f}')
print(f'Test F1:  {f1_score(y_test, best_xgb.predict(X_test_enc)):.4f}')

## 5 — Optuna con LightGBM

In [ ]:
def objective_lgb(trial):
    params = {
        'n_estimators':       1000,
        'learning_rate':      trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'num_leaves':         trial.suggest_int('num_leaves', 20, 300),
        'min_child_samples':  trial.suggest_int('min_child_samples', 5, 100),
        'feature_fraction':   trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction':   trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'bagging_freq':       trial.suggest_int('bagging_freq', 1, 10),
        'reg_alpha':          trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'random_state': 42, 'verbose': -1,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
    )
    y_prob = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_prob)

study_lgb = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_lgb.optimize(objective_lgb, n_trials=60, show_progress_bar=False)

print(f'Mejor Val AUC (LightGBM): {study_lgb.best_value:.4f}')

best_lgb_params = {**study_lgb.best_params, 'n_estimators': 1000,
                   'random_state': 42, 'verbose': -1}
best_lgb = lgb.LGBMClassifier(**best_lgb_params)
best_lgb.fit(
    X_tr, y_tr, eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
)
print(f'Test AUC: {roc_auc_score(y_test, best_lgb.predict_proba(X_test_enc)[:,1]):.4f}')

## 6 — Visualizaciones de Optuna

In [ ]:
# Historia de optimización: cómo mejora el best value a lo largo de los trials
fig, ax = plt.subplots(figsize=(10, 4))
plot_optimization_history(study_xgb, ax=ax)
ax.set_title('XGBoost — Optimization History (TPE)')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia de hiperparámetros según Optuna (FAnova)
fig, ax = plt.subplots(figsize=(8, 5))
plot_param_importances(study_xgb, ax=ax)
ax.set_title('Importancia de hiperparámetros — XGBoost (FAnova)')
plt.tight_layout()
plt.show()
print('FAnova estima qué fracción de la varianza del score se explica por cada hiperparámetro.')

In [ ]:
# Slice plot: cómo varía el score con cada hiperparámetro individualmente
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
params_to_plot = ['learning_rate', 'max_depth', 'subsample',
                  'colsample_bytree', 'reg_alpha', 'min_child_weight']
plot_slice(study_xgb, params=params_to_plot, axes=axes.flatten()[:len(params_to_plot)])
plt.suptitle('XGBoost — Slice Plot: AUC vs cada hiperparámetro', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 7 — Espacio condicional: selección de modelo dentro de Optuna

In [ ]:
# Optuna puede buscar sobre distintos modelos en el mismo study
def objective_model_selection(trial):
    model_type = trial.suggest_categorical('model', ['decision_tree', 'random_forest', 'xgboost', 'lightgbm'])

    if model_type == 'decision_tree':
        model = DecisionTreeClassifier(
            max_depth=trial.suggest_int('dt_max_depth', 2, 12),
            min_samples_leaf=trial.suggest_int('dt_min_leaf', 1, 40),
            ccp_alpha=trial.suggest_float('dt_ccp_alpha', 0.0, 0.05),
            random_state=42
        )
        scores = cross_val_score(model, X_train_enc, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    elif model_type == 'random_forest':
        model = RandomForestClassifier(
            n_estimators=trial.suggest_int('rf_n_est', 50, 400),
            max_depth=trial.suggest_int('rf_max_depth', 3, 15),
            min_samples_leaf=trial.suggest_int('rf_min_leaf', 1, 20),
            max_features=trial.suggest_categorical('rf_max_feat', ['sqrt', 'log2', 0.5]),
            random_state=42, n_jobs=-1
        )
        scores = cross_val_score(model, X_train_enc, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
        return scores.mean()

    elif model_type == 'xgboost':
        model = xgb.XGBClassifier(
            n_estimators=1000,
            learning_rate=trial.suggest_float('xgb_lr', 1e-3, 0.3, log=True),
            max_depth=trial.suggest_int('xgb_depth', 3, 9),
            subsample=trial.suggest_float('xgb_sub', 0.5, 1.0),
            colsample_bytree=trial.suggest_float('xgb_col', 0.5, 1.0),
            early_stopping_rounds=30, eval_metric='auc',
            random_state=42, verbosity=0
        )
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        return model.best_score

    else:  # lightgbm
        model = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=trial.suggest_float('lgb_lr', 1e-3, 0.3, log=True),
            num_leaves=trial.suggest_int('lgb_leaves', 20, 200),
            min_child_samples=trial.suggest_int('lgb_min_child', 5, 80),
            random_state=42, verbose=-1
        )
        model.fit(
            X_tr, y_tr, eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(period=-1)]
        )
        return roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

study_sel = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study_sel.optimize(objective_model_selection, n_trials=80, show_progress_bar=False)

print(f'Mejor AUC global: {study_sel.best_value:.4f}')
print(f'Mejor modelo:     {study_sel.best_params["model"]}')
print('Parámetros del ganador:')
for k, v in study_sel.best_params.items():
    print(f'  {k:<30} = {v:.6f}' if isinstance(v, float) else f'  {k:<30} = {v}')

# Cuántos trials por modelo?
trials_df = study_sel.trials_dataframe()
print('\nTrials por modelo:')
print(trials_df['params_model'].value_counts().to_string())

## 8 — Pruning: parar trials malos antes de terminar

In [ ]:
# Con pruning manual: reportar métricas intermedias para que Optuna corte temprano
# Útil cuando el entrenamiento es largo (e.g., muchas épocas)

from sklearn.model_selection import StratifiedKFold

def objective_with_pruning(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 50, 500),
        'max_depth':        trial.suggest_int('max_depth', 3, 12),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 30),
        'max_features':     trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    auc_scores = []

    for step, (tr_idx, val_idx) in enumerate(cv.split(X_train_enc, y_train)):
        X_tr_cv, X_val_cv = X_train_enc[tr_idx], X_train_enc[val_idx]
        y_tr_cv, y_val_cv = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        model = RandomForestClassifier(**params, random_state=42, n_jobs=-1)
        model.fit(X_tr_cv, y_tr_cv)
        score = roc_auc_score(y_val_cv, model.predict_proba(X_val_cv)[:, 1])
        auc_scores.append(score)

        # Reportar métrica intermedia (fold completado) y preguntar si podar
        trial.report(np.mean(auc_scores), step)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(auc_scores)

study_pruned = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=2)
)
study_pruned.optimize(objective_with_pruning, n_trials=60, show_progress_bar=False)

pruned = [t for t in study_pruned.trials if t.state == optuna.trial.TrialState.PRUNED]
complete = [t for t in study_pruned.trials if t.state == optuna.trial.TrialState.COMPLETE]
print(f'Trials completados: {len(complete)} | Podados: {len(pruned)}')
print(f'Mejor AUC: {study_pruned.best_value:.4f}')
print('Mejores parámetros:')
for k, v in study_pruned.best_params.items():
    print(f'  {k:<25} = {v}')

## 9 — Comparar Optuna vs GridSearchCV

In [ ]:
import time
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# GridSearchCV
param_grid = {
    'max_depth':        [3, 5, 7, 10],
    'min_samples_leaf': [1, 5, 10, 20],
    'max_features':     ['sqrt', 'log2'],
    'n_estimators':     [100, 200],
}
# 4 * 4 * 2 * 2 = 64 combinaciones

t0 = time.time()
gs = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1
)
gs.fit(X_train_enc, y_train)
t_gs = time.time() - t0

# Optuna (mismo número de trials que combinaciones en grid)
def objective_rf(trial):
    return cross_val_score(
        RandomForestClassifier(
            n_estimators=trial.suggest_categorical('n_estimators', [100, 200]),
            max_depth=trial.suggest_int('max_depth', 3, 10),
            min_samples_leaf=trial.suggest_int('min_samples_leaf', 1, 20),
            max_features=trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            random_state=42, n_jobs=-1
        ),
        X_train_enc, y_train, cv=5, scoring='roc_auc', n_jobs=-1
    ).mean()

t0 = time.time()
study_rf = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=64, show_progress_bar=False)
t_opt = time.time() - t0

print(f'{"Método":<22} {"AUC":>8}  {"Tiempo":>10}  {"N trials":>10}')
print('-' * 56)
gs_auc  = roc_auc_score(y_test, gs.best_estimator_.predict_proba(X_test_enc)[:, 1])
print(f'{"GridSearchCV (64)":<22} {gs.best_score_:>8.4f}  {t_gs:>9.1f}s  {64:>10}')
print(f'{"Optuna (64 trials)":<22} {study_rf.best_value:>8.4f}  {t_opt:>9.1f}s  {64:>10}')

# Optuna con menos trials
t0 = time.time()
study_rf_30 = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_rf_30.optimize(objective_rf, n_trials=30, show_progress_bar=False)
t_opt30 = time.time() - t0
print(f'{"Optuna (30 trials)":<22} {study_rf_30.best_value:>8.4f}  {t_opt30:>9.1f}s  {30:>10}')

print('\n→ Optuna con 30 trials suele igualar o superar GridSearch con 64 por ser dirigido.')

## 10 — Guardar y retomar un Study

In [ ]:
# Persistir el study en SQLite para poder retomarlo o compartirlo
storage = 'sqlite:////private/tmp/optuna_example.db'
study_name = 'xgb_conversion_study'

# create_study con storage → persiste cada trial; load_if_exists permite retomar
study_persistent = optuna.create_study(
    study_name=study_name,
    storage=storage,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42),
    load_if_exists=True
)
study_persistent.optimize(objective_xgb, n_trials=20, show_progress_bar=False)

print(f'Study: {study_name}')
print(f'Trials totales: {len(study_persistent.trials)}')
print(f'Mejor AUC:      {study_persistent.best_value:.4f}')

# Para cargar en otra sesión:
print('\nPara retomar en otra sesión:')
print(f"  study = optuna.load_study(study_name='{study_name}', storage='{storage}')")
print(f"  study.optimize(objective, n_trials=50)  # continúa desde donde paró")

## Resumen — Flujo recomendado con Optuna

```python
# 1. Definir objective function con trial.suggest_*
def objective(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'max_depth':     trial.suggest_int('max_depth', 3, 10),
    }
    model = XGBClassifier(**params, n_estimators=1000, early_stopping_rounds=30)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return model.best_score

# 2. Crear y optimizar
study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=100)

# 3. Reentrenar con mejores params
best_model = XGBClassifier(**study.best_params)
best_model.fit(X_train, y_train)
```

| Situación | Recomendación |
|---|---|
| Espacio pequeño (< 100 combos) | `GridSearchCV` — exhaustivo y predecible |
| Espacio grande, modelos rápidos | Optuna con CV, 50–150 trials |
| Modelos lentos (GBTs con early stopping) | Optuna con hold-out val + pruning |
| Selección de modelo + hiperparámetros | Optuna con espacio condicional |
| Varios procesos / máquinas | Optuna con storage SQLite/PostgreSQL |

**Referencias relacionadas:** `04b_cart_decision_trees.ipynb` · `09_ensemble_methods.ipynb` · `09b_gradient_boosting_frameworks.ipynb`